# SHAPE — Notebook 2 of 4 · Agent & ReAct Engine

**Group 3 | ITAI2376-14562**  
Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas

**Coverage (rubric):** Memory system · Reasoning & ReAct · Output generation  
**Imports from:** `shape.parser`, `shape.memory`, `shape.prompts`, `shape.agent`


In [ ]:
# Shared setup — identical across all notebooks
import os, sys
from pathlib import Path
if not Path('shape').exists() and Path('../shape').exists():
    os.chdir('..')
root = Path('.').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('Root:', root, '| shape/ exists:', Path('shape').exists())

## 1. Prompt Construction (`shape.prompts`)

The system prompt is a template that fills in two values at runtime:
- `{tool_descriptions}` — one line per tool from the registry
- `{tool_names}` — comma-separated list (constrains the LLM's Action: choices)

An RL policy hint is appended after the template if `FeedbackMemory.get_policy_hint()` returns a non-empty string. This is the **only** mechanism by which past feedback changes agent behaviour.


In [ ]:
from shape.prompts import REACT_SYSTEM
from shape.tools   import get_tools

config = {
    'executor_timeout': 10, 'executor_max_output': 4096,
    'workspace_dir': 'workspace', 'doc_index_path': 'data/doc_index',
    'docs_dir': 'data/docs', 'doc_top_k': 3,
}
tools = get_tools(config)
descriptions = '\n'.join(f'- {t.name}: {t.description}' for t in tools)
names        = ', '.join(t.name for t in tools)
filled       = REACT_SYSTEM.format(tool_descriptions=descriptions, tool_names=names)

print(f'System prompt ({len(filled)} chars, {len(filled.split())} words):')
print('=' * 60)
print(filled)

## 2. Parser — `shape.parser.parse_llm_output()`

Because `ChatOllama` has no native tool-calling API, we parse the LLM's
plain text output with `re.search`. Three branches in priority order:

1. **Final Answer** — `re.search(r"Final Answer\s*:\s*(.+)", text, DOTALL)`
2. **Action** — `re.search(r"Action\s*:\s*(.+)", text)`  
3. **Fallback** — treat entire text as a final answer (never crashes)

This is the most critical piece of SHAPE's architecture — it is what turns
a plain text completion model into an agent that uses tools.


In [ ]:
from shape.parser import parse_llm_output
import inspect

# Show the actual source code
print('source: shape/parser.py')
print('=' * 60)
print(inspect.getsource(parse_llm_output))
print()

# Test all three branches
cases = [
    ('Thought: Lint first.\nAction: linter\nAction Input: def f(): pass',
     'Branch 2 — Action'),
    ('Thought: Done.\nFinal Answer: def fib(n): return n if n<2 else fib(n-1)+fib(n-2)',
     'Branch 1 — Final Answer'),
    ('Here is a solution: print("hello")',
     'Branch 3 — Fallback'),
    ('Action: web_browser\nAction Input: search',
     'Unknown tool (hallucination) — still parses cleanly'),
]

for text, label in cases:
    r = parse_llm_output(text)
    print(f'{label}')
    print(f'  input:  {repr(text[:60])}')
    if r['type'] == 'action':
        print(f'  result: type=action, tool={r["tool"]!r}, input={repr(r["input"][:40])}')
    else:
        print(f'  result: type=final, answer={repr(r["answer"][:60])}')
    print()

## 3. Memory — `shape.memory`

`InMemoryChatMessageHistory` stores the conversation as a flat list of messages.
`trimmed_messages(history, window)` slices the last `window * 2` entries so the
context window budget is never exceeded (see token budget analysis in NB01 appendix).

Memory persists within a session but resets on `agent.reset_session()`.
Across reboots the conversation is not persisted — only the RL feedback JSONL is.


In [ ]:
from shape.memory import make_history, trimmed_messages

h = make_history()
for i in range(10):
    h.add_user_message(f'question {i}')
    h.add_ai_message(f'answer {i}')

print(f'Total messages stored: {len(h.messages)}')
print()

print(f'{"Window":<8} {"Messages trimmed to":<22} {"Token budget saved"}')
print('-' * 60)
avg_tokens_per_msg = 90  # empirical estimate on Pi 5
for w in [2, 4, 6, 8, 10]:
    trimmed = trimmed_messages(h, w)
    saved   = (20 - len(trimmed)) * avg_tokens_per_msg
    marker  = '  <-- config default' if w == 6 else ''
    print(f'  {w:<6}  {len(trimmed):<22} {saved} tokens{marker}')

In [ ]:
# Initialize the agent — single call, shared by this notebook and NB04.
# verbose=True shows every ReAct step in the output cells.

import logging, os
logging.basicConfig(level=logging.WARNING)
os.makedirs('data', exist_ok=True)
os.makedirs('workspace', exist_ok=True)

from shape.agent import build_agent

AGENT_CONFIG = {
    'model':            'codellama',
    'ollama_url':       'http://localhost:11434',
    'temperature':      0.2,
    'context_window':   4096,
    'memory_window':    6,
    'max_iterations':   8,
    'executor_timeout': 10,
    'feedback_path':    'data/feedback.jsonl',
    'workspace_dir':    'workspace',
    'doc_index_path':   'data/doc_index',
    'docs_dir':         'data/docs',
    'doc_top_k':        3,
    'verbose':          True,
}

agent = build_agent(AGENT_CONFIG)

print('Agent components:')
print(f'  LLM:         {type(agent.llm).__name__}')
print(f'  Tools:       {list(agent.tools_by_name)}')
print(f'  Parser:      shape.parser.parse_llm_output (shared module)')
print(f'  Memory:      {type(agent._history).__name__}')
print(f'  Rewards:     {type(agent.feedback).__name__}')
print(f'  Safety:      {type(agent.guard).__name__}')

In [ ]:
# Live ReAct loop — observe the full Thought/Action/Observation trace
# verbose=True in AGENT_CONFIG makes each step print automatically.

print('=' * 65)
print('QUERY: palindrome checker with linter + executor')
print('=' * 65)

response = agent.run(
    'Write a Python function that checks whether a string is a palindrome, '
    'ignoring case and spaces. Use the linter first, then run it with '
    'three test cases including a multi-word phrase.'
)

print('\n' + '=' * 65)
print('FINAL ANSWER')
print('=' * 65)
print(response)

In [ ]:
# Prove memory: follow-up should reference the palindrome function without
# the user re-describing it.

response2 = agent.run(
    'Now extend that same function to accept a list of strings and return '
    'a dict mapping each string to True/False.'
)
print(response2)

print(f'\nHistory buffer: {len(agent._history.messages)} messages '
      f'(window={AGENT_CONFIG["memory_window"]} turns)')

In [ ]:
# 8-step cap: when max_iterations is exhausted, the fallback string is
# returned — never an exception, never a hang.

from shape.agent import build_agent
capped = build_agent({**AGENT_CONFIG, 'max_iterations': 1, 'verbose': False})
result = capped.run('Write bubble sort and test it.')
print(f'With max_iterations=1: {repr(result[:100])}')
print('(Either a one-shot Final Answer or the fallback — never an exception)')

## Summary

| Concept | Module | Demonstrated |
|---|---|---|
| Prompt construction | `shape.prompts` | REACT_SYSTEM template filled live |
| ReAct parser | `shape.parser` | All 3 branches + source code shown |
| Memory trimming | `shape.memory` | Window budget table, follow-up works |
| Agent init | `shape.agent` | Single `build_agent()` call |
| Full ReAct trace | `shape.agent.run()` with verbose | Thought/Action/Obs printed |
| 8-step cap | `max_iterations=1` test | Fallback string, no exception |

▶ **Continue to `03_SHAPE_Tools_Environment.ipynb`**
